# AegisRAG · Stage 00 — Synthetic Data Generation
    Generates all six training artefacts:
    * `qa_pairs.jsonl`, `preferences.jsonl`, `confidence_labels.jsonl`, `alpha_labels.jsonl`, `decomp_labels.jsonl`, `tool_route_labels.jsonl`
    Requires only the `aegisrag-source` + `aegisrag-raw-docs` Kaggle Datasets.
    Outputs land in `/kaggle/working/data/synthetic/*.jsonl` — save as a new Kaggle Dataset named **aegisrag-synthetic** and attach to later notebooks.


In [ ]:
# ---- Kaggle bootstrap ----------------------------------------------------
# Attach these Kaggle Datasets to the notebook before running:
#   aegisrag-source       (snapshot of the git repo: src/, config/, run.py, requirements.txt)
#   aegisrag-raw-docs     (raw customer-support documents)            [only for data-gen]
#   aegisrag-synthetic    (generated qa/preferences/labels .jsonl)    [all training stages]
#   aegisrag-checkpoints  (prior-stage checkpoints)                   [required for DPO]
import sys, os, importlib.util
spec = importlib.util.spec_from_file_location(
    "aegisrag_bootstrap",
    "/kaggle/input/aegisrag-source/kaggle/helpers/bootstrap.py",
)
bootstrap = importlib.util.module_from_spec(spec); spec.loader.exec_module(bootstrap)
bootstrap.setup_kaggle(
    component='data_gen',
    install_train_deps=False,
    install_parse_deps=True,
)


In [ ]:
# Route synthetic output to /kaggle/working and chunking input to /kaggle/input.
import os
os.environ["AEGIS_DATA__SYNTHETIC_DIR"] = "/kaggle/working/data/synthetic"
os.environ["AEGIS_DATA__SYNTHETIC__QA_PATH"] = "/kaggle/working/data/synthetic/qa_pairs.jsonl"
os.environ["AEGIS_DATA__SYNTHETIC__PREFERENCES_PATH"] = "/kaggle/working/data/synthetic/preferences.jsonl"
os.environ["AEGIS_DATA__SYNTHETIC__CONFIDENCE_LABELS_PATH"] = "/kaggle/working/data/synthetic/confidence_labels.jsonl"
os.environ["AEGIS_DATA__SYNTHETIC__ALPHA_LABELS_PATH"] = "/kaggle/working/data/synthetic/alpha_labels.jsonl"
os.environ["AEGIS_DATA__SYNTHETIC__DECOMP_LABELS_PATH"] = "/kaggle/working/data/synthetic/decomp_labels.jsonl"
os.environ["AEGIS_DATA__SYNTHETIC__TOOL_ROUTE_LABELS_PATH"] = "/kaggle/working/data/synthetic/tool_route_labels.jsonl"

# Reload config to pick up env overrides.
from src.utils.config import reset_config, get_config
reset_config(); cfg = get_config()
print("synthetic_dir:", cfg.data.synthetic_dir)


In [ ]:
# 1) Ingest raw docs (chunking + BM25/Chroma indices in /kaggle/working).
from src.data.ingestion import run_ingestion
stats = run_ingestion(source_dir=cfg.data.raw_docs_dir)
stats


In [ ]:
# 2) Generate all six synthetic artefacts.
from scripts.generate_data import main as gen_main
import argparse
gen_main(argparse.Namespace(type="all", output_dir=cfg.data.synthetic_dir, limit=None))


In [ ]:
# ---- Persist outputs -----------------------------------------------------
# Everything under /kaggle/working is captured as the notebook's output and
# can be saved as a new Kaggle Dataset via File > "Save Version"
# with output = "Save & Run All (Commit)".
import shutil, os
from pathlib import Path
src_dir = Path("/kaggle/working/checkpoints/../data/synthetic")
print("Synthetic data:", src_dir, "->", os.listdir(src_dir) if src_dir.exists() else "(missing)")
